# RecursiveGCD (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_java.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `RecursiveGCD`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/RecursiveGCD_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Massimo comun divisore: RecursiveGCD

**Input:** due numeri naturali $n \ge 1$ e $m \ge 0$. **Output:** $\gcd(n, m)$. Ricorsione di coda: ogni chiamata ne fa al più una, l'albero è un cammino. Al più $2\lfloor\log_2 m\rfloor + 3$ chiamate, $O(\log_2 m)$ passi.

Pseudocodice: sezione 14.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
%%writefile RecursiveGCD.java
public class RecursiveGCD
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int m)                    // floor(log2 m) con un ciclo (solo per il limite sulle chiamate)
    {
        int k = 0;
        while (m > 1)
        {
            m = m / 2;
            k = k + 1;
        }
        return k;
    }

    // RecursiveGCD(n, m)
    static int recursiveGCD(int n, int m)
    {
        push(m); cnt[2]++;
        if (m == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("caso base, restituisce %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("%d mod %d = %d: chiama RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
        cnt[5]++;
        int g = recursiveGCD(m, n % m);
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("restituisce %d", g));
        pop();
        return g;
    }

    public static void main(String[] args)
    {
        // Esempi delle dispense: RecursiveGCD(48, 18) e RecursiveGCD(252, 105), alberi delle chiamate (cammini)
        int[][] pairs = {{48, 18}, {252, 105}};
        for (int[] p : pairs)
        {
            reset();
            trace = true;
            System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
            int g = recursiveGCD(p[0], p[1]);
            trace = false;
            System.out.printf("gcd(%d, %d) = %d, chiamate = %d, S = %d\n", p[0], p[1], g, calls, total());
            System.out.printf("\n");
        }

        // Numero di chiamate (ogni chiamata esegue 2 righe, quindi S = 2 chiamate) e limite 2 floor(log2 m) + 3
        System.out.printf("      n |       m |  gcd | chiamate |   S | 2 floor(log2 m) + 3\n");
        int[][] more = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
        for (int[] p : more)
        {
            reset();
            int g = recursiveGCD(p[0], p[1]);
            System.out.printf("%7d | %7d | %4d | %8d | %3d | %d\n", p[0], p[1], g, calls, total(), 2 * floorLog2(p[1]) + 3);
        }
    }
}

Writing RecursiveGCD.java


In [3]:
!javac RecursiveGCD.java && java RecursiveGCD

chiamata                    | azione
RecursiveGCD(48, 18)        | 48 mod 18 = 12: chiama RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: chiama RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: chiama RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | caso base, restituisce 6
    RecursiveGCD(12, 6)     | restituisce 6
  RecursiveGCD(18, 12)      | restituisce 6
RecursiveGCD(48, 18)        | restituisce 6
gcd(48, 18) = 6, chiamate = 4, S = 8

chiamata                    | azione
RecursiveGCD(252, 105)      | 252 mod 105 = 42: chiama RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: chiama RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(42, 21)    | restituisce 21
  RecursiveGCD(105, 42)     | restituisce 21
RecursiveGCD(252, 105)      | restituisce 21
gcd(252, 105) = 21, chiamate = 4, S = 8

      n |       m |  gcd | ch

**Prova tu.** Cambia i due numeri; il caso peggiore sono due numeri di Fibonacci consecutivi.

In [4]:
%%writefile ProvaRecursiveGCD.java
public class ProvaRecursiveGCD
{
    public static void main(String[] args)
    {
        int n = 1071, m = 462;        // <-- cambia i due numeri
        RecursiveGCD.reset();
        RecursiveGCD.trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        int g = RecursiveGCD.recursiveGCD(n, m);
        RecursiveGCD.trace = false;
        System.out.printf("gcd(%d, %d) = %d, chiamate = %d\n", n, m, g, RecursiveGCD.calls);
    }
}

Writing ProvaRecursiveGCD.java


In [5]:
!javac ProvaRecursiveGCD.java && java ProvaRecursiveGCD

chiamata                    | azione
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: chiama RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: chiama RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(147, 21)   | restituisce 21
  RecursiveGCD(462, 147)    | restituisce 21
RecursiveGCD(1071, 462)     | restituisce 21
gcd(1071, 462) = 21, chiamate = 4


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).